# Stage 1: Base Model SFT & Stage 2: Activation Extraction

This notebook coordinates the parameter-efficient supervised fine-tuning (LoRA SFT) of the base model using Unsloth on Kaggle, followed by layer activation extraction and chunked memory-safe caching.

## 1. Setup Environment & Dependencies

Install the optimized libraries for training. On Kaggle, Unsloth must be installed using their specific wheels or git repository.

In [ ]:
# Install Unsloth and standard training dependencies
!pip install -q lightning pytorch-lightning transformers accelerate safetensors datasets trl
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Supervised Fine-Tuning (SFT)

We run the training script `src/system1/train_sft.py`. For demonstration or quick checks, you can use the `--debug` flag to train on a small subset of the dataset.

In [ ]:
# Run fine-tuning on DeepSeek-R1-Distill-Qwen-1.5B
# Note: Remove the --debug flag to run full SFT on the entire dataset
!python src/system1/train_sft.py \
    --model_name "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" \
    --dataset "tatsu-lab/alpaca" \
    --output_dir "./outputs" \
    --adapter_dir "./adapters" \
    --batch_size 2 \
    --gradient_accumulation_steps 4 \
    --epochs 1 \
    --debug

## 3. Batched Activation Extraction & Chunked Caching

Run `src/system1/extract_activations.py` to hook Layer 14 of the fine-tuned base model, run batched inference, and write the continuous activations to disk in chunked `.safetensors` files to prevent RAM overflow.

In [ ]:
# Extract activations using the fine-tuned LoRA adapters
# Caches output in files of 100 samples each to ./cached_activations/
!python src/system1/extract_activations.py \
    --model_name "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" \
    --adapter_dir "./adapters" \
    --dataset "tatsu-lab/alpaca" \
    --output_dir "./cached_activations" \
    --layer_index 14 \
    --batch_size 2 \
    --chunk_size 100 \
    --debug

## 4. Verify Cached Activations

We can inspect the output directory to verify that chunk files exist and load the first chunk to check its shape.

In [ ]:
import os
from safetensors.torch import load_file

cache_dir = "./cached_activations"
chunks = [f for f in os.listdir(cache_dir) if f.endswith(".safetensors")]
print(f"Found {len(chunks)} cached chunk files in {cache_dir}:")
for chunk in sorted(chunks):
    filepath = os.path.join(cache_dir, chunk)
    data = load_file(filepath)
    print(f"  {chunk}: shape {data['activations'].shape}")